In [48]:
import pandas as pd
from pyprojroot import here

In [49]:
RAW_DATA_DIR = here() / "data" / "raw"
INTERIM_DATA_DIR = here() / "data" / "interim"

In [50]:
c2020 = pd.read_csv(RAW_DATA_DIR/"population/census_2020/conjunto_de_datos_iter_00CSV20.csv", low_memory=False)
c2010 = pd.read_csv(RAW_DATA_DIR/"population/census_2010/iter_00_cpv2010.csv")

In [51]:
c2020['CVEGEO'] = c2020['ENTIDAD'] * 1000 + c2020['MUN']
c2010['CVEGEO'] = c2010['entidad'] * 1000 + c2010['mun']

In [52]:
c2020_redux = c2020[['CVEGEO', 'POBTOT']].copy()
c2010_redux = c2010[['CVEGEO', 'pobtot']].copy()

In [53]:
c2020_redux.rename(columns={'POBTOT': '2020', 'anio': 'year'}, inplace=True)
c2010_redux.rename(columns={'pobtot': '2010', 'anio': 'year'}, inplace=True)

In [54]:
cdf = pd.merge(cdf2010, cdf2020, on=('CVEGEO'))

In [55]:
cdf2025 = cdf.set_index('CVEGEO').copy()
cdf2025['grw'] = (cdf2025['2020'] - cdf2025['2010'])/10
for anio in range(2011, 2020):
    cdf2025[anio] = cdf2025['2010'] + cdf2025['grw'] * (anio - 2010)

for anio in range(2021, 2026):
    cdf2025[anio] = cdf2025['2020'] + cdf2025['grw'] * (anio - 2020)

cdf2025 = cdf2025.drop(columns='grw').reset_index()

result = cdf2025.melt(id_vars='CVEGEO', var_name='year', value_name='population')

In [56]:
result.to_csv(INTERIM_DATA_DIR/"population_2015_2025.csv")

print(f" {len(result)} municipalities processed")
print(result.sample(10))

 39824 municipalities processed
       CVEGEO  year  population
986     19020  2010     11413.0
33015   14113  2023     33935.3
18693   20252  2016      4553.6
15109    7088  2015      4560.0
36811   26051  2024      9460.0
18911   20470  2016      9073.8
21849   26023  2017     12001.3
12581    7049  2014     49429.0
7170    30082  2011     30094.7
4644    30045  2020    109617.0
